In [ ]:
#============================================================
# Celda 1 — Ingesta y procesado EVR por sexo y provincia
#============================================================
import os, json
import pandas as pd
import numpy as np
# ROOT dinámico — funciona en local, CI y cualquier entorno
ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT)

from pathlib import Path

OUTPUT = Path("output")
FIGS   = OUTPUT / "09_analisis_genero" / "figs"
TABLES = OUTPUT / "09_analisis_genero" / "tables"
FIGS.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

# Cargar JSON
with open("data/evr/evr_flujos_provincia_sexo.json", encoding="utf-8-sig") as f:
    raw = json.load(f)

# Parsear: sexo, provincia, año, valor
registros = []
for serie in raw:
    nombre = serie["Nombre"]
    partes  = [p.strip() for p in nombre.split(".")]

    # Filtrar solo Hombres/Mujeres (ignorar Total) con provincia específica
    if partes[0] not in ("Hombres", "Mujeres"):
        continue
    if len(partes) < 2 or partes[1] == "Total Nacional":
        continue

    sexo      = partes[0]
    provincia = partes[1]

    for dato in serie["Data"]:
        if dato["Anyo"] is None or dato["Valor"] is None:
            continue
        registros.append({
            "sexo":      sexo,
            "provincia": provincia,
            "anyo":      int(dato["Anyo"]),
            "flujo":     float(dato["Valor"]),
        })

df_raw = pd.DataFrame(registros)
print(f"Registros crudos: {df_raw.shape}")
print(df_raw.head(8))
print(f"\nAños: {sorted(df_raw['anyo'].unique())}")
print(f"Sexos: {df_raw['sexo'].unique()}")
print(f"Provincias: {df_raw['provincia'].nunique()}")

In [ ]:
#============================================================
# Celda 2 — Pivot y cálculo de métricas de género
#============================================================
# Filtrar años del estudio
df = df_raw[df_raw["anyo"].between(2013, 2021)].copy()

# Pivot: una fila por (provincia, anyo), columnas Hombres/Mujeres
pivot = df.pivot_table(
    index=["provincia", "anyo"],
    columns="sexo",
    values="flujo",
    aggfunc="sum"
).reset_index()

pivot.columns.name = None
pivot = pivot.rename(columns={"Hombres": "flujo_hombres", "Mujeres": "flujo_mujeres"})

# Métricas derivadas
pivot["flujo_total"]        = pivot["flujo_hombres"] + pivot["flujo_mujeres"]
pivot["ratio_feminizacion"] = pivot["flujo_mujeres"] / pivot["flujo_total"] * 100
pivot["brecha_absoluta"]    = pivot["flujo_mujeres"] - pivot["flujo_hombres"]

# Merge con master para añadir saldo_neto y resto de variables
master = pd.read_parquet("output/merged/master_provincia_anio.parquet")
df_gen = pivot.merge(master[["provincia", "anyo", "saldo_neto", "tasa_atraccion",
                               "cob_100mbps_pct", "compraventas_per_capita"]],
                     on=["provincia", "anyo"], how="inner")

print(f"✅ Dataset género: {df_gen.shape}")
print(df_gen.head(6).to_string(index=False))

# Guardar
df_gen.to_parquet(OUTPUT / "09_analisis_genero" / "genero_provincia_anio.parquet", index=False)
df_gen.to_csv(TABLES / "genero_provincia_anio.csv", index=False)
print("\n✅ Guardado: genero_provincia_anio.parquet y .csv")

In [ ]:
#============================================================
# Celda 3 — Setup visualización
#============================================================
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

plt.rcParams.update({
    "figure.dpi":        150,
    "figure.facecolor":  "white",
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.family":       "DejaVu Sans",
    "axes.titlesize":    12,
    "axes.labelsize":    10,
})

C_H   = "#2980b9"   # azul — hombres
C_M   = "#e74c3c"   # rojo — mujeres
C_TOT = "#27ae60"   # verde — total/ratio
C_NEU = "#7f8c8d"   # gris neutro

print("✅ Setup listo")

In [ ]:
#============================================================
# Celda 4 — Evolución nacional flujos Hombres vs Mujeres
#============================================================
nac = df_gen.groupby("anyo")[["flujo_hombres", "flujo_mujeres"]].sum().reset_index()

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(nac["anyo"], nac["flujo_hombres"], marker="o", color=C_H,
        linewidth=2.5, label="Hombres")
ax.plot(nac["anyo"], nac["flujo_mujeres"], marker="o", color=C_M,
        linewidth=2.5, label="Mujeres")

ax.fill_between(nac["anyo"], nac["flujo_hombres"], nac["flujo_mujeres"],
                alpha=0.08, color=C_NEU, label="Brecha")

ax.axvline(2020, color="#8e44ad", linewidth=1.5, linestyle="--", alpha=0.7)
ax.text(2020.1, ax.get_ylim()[1] * 0.96, "COVID", color="#8e44ad", fontsize=8)

ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_xticks(nac["anyo"])
ax.set_title("Flujos migratorios interprovinciales por sexo (2013–2021)\nEspaña agregado nacional")
ax.set_ylabel("Personas en movimiento")
ax.legend()

plt.tight_layout()
plt.savefig(FIGS / "01_evolucion_flujos_sexo_nacional.png")
plt.show()
print("✅ 01_evolucion_flujos_sexo_nacional.png")

In [ ]:
#============================================================
# Celda 5 — Ratio feminización nacional por año
#============================================================
nac["ratio_fem"] = nac["flujo_mujeres"] / (nac["flujo_hombres"] + nac["flujo_mujeres"]) * 100

fig, ax = plt.subplots(figsize=(10, 5))
colores = [C_M if r >= 50 else C_H for r in nac["ratio_fem"]]
bars = ax.bar(nac["anyo"], nac["ratio_fem"], color=colores, edgecolor="white", width=0.6)

ax.axhline(50, color="black", linewidth=1.2, linestyle="--", label="Paridad (50%)")
ax.set_ylim(45, 55)
ax.set_xticks(nac["anyo"])
ax.set_ylabel("% mujeres sobre total flujo")
ax.set_title("Ratio de feminización de la migración interprovincial (2013–2021)")
ax.legend()

for bar, v in zip(bars, nac["ratio_fem"]):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.1,
            f"{v:.1f}%", ha="center", va="bottom", fontsize=8)

plt.tight_layout()
plt.savefig(FIGS / "02_ratio_feminizacion_nacional.png")
plt.show()
print("✅ 02_ratio_feminizacion_nacional.png")

In [ ]:
#============================================================
# Celda 6 — Ranking provincias por ratio feminización (media 2013–2021)
#============================================================
media_prov = (df_gen.groupby("provincia")["ratio_feminizacion"]
              .mean().reset_index()
              .sort_values("ratio_feminizacion", ascending=False))

top15_fem  = media_prov.head(15)
top15_masc = media_prov.tail(15).sort_values("ratio_feminizacion")

fig, axes = plt.subplots(1, 2, figsize=(15, 7))

# Más feminizadas
axes[0].barh(top15_fem["provincia"], top15_fem["ratio_feminizacion"],
             color=C_M, edgecolor="white")
axes[0].axvline(50, color="black", linewidth=0.8, linestyle="--")
axes[0].set_xlim(45, 56)
axes[0].set_title("Top 15 provincias\nmayor feminización migratoria", color=C_M)
axes[0].set_xlabel("% mujeres sobre flujo total")
for i, v in enumerate(top15_fem["ratio_feminizacion"]):
    axes[0].text(v + 0.1, i, f"{v:.1f}%", va="center", fontsize=8)

# Más masculinizadas
axes[1].barh(top15_masc["provincia"], top15_masc["ratio_feminizacion"],
             color=C_H, edgecolor="white")
axes[1].axvline(50, color="black", linewidth=0.8, linestyle="--")
axes[1].set_xlim(42, 51)
axes[1].set_title("Top 15 provincias\nmayor masculinización migratoria", color=C_H)
axes[1].set_xlabel("% mujeres sobre flujo total")
for i, v in enumerate(top15_masc["ratio_feminizacion"]):
    axes[1].text(v + 0.1, i, f"{v:.1f}%", va="center", fontsize=8)

plt.suptitle("Ratio de feminización de la migración interprovincial — media 2013–2021",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(FIGS / "03_ranking_feminizacion_provincias.png", bbox_inches="tight")
plt.show()
print("✅ 03_ranking_feminizacion_provincias.png")

In [ ]:
#============================================================
# Celda 7 — Heatmap ratio feminización por provincia y año
#============================================================
pivot_fem = df_gen.pivot_table(
    index="provincia", columns="anyo",
    values="ratio_feminizacion", aggfunc="mean"
)
pivot_fem = pivot_fem.loc[pivot_fem.mean(axis=1).sort_values(ascending=False).index]

fig, ax = plt.subplots(figsize=(13, 16))
sns.heatmap(pivot_fem, ax=ax, cmap="RdBu", center=50,
            linewidths=0.3, linecolor="#dddddd", annot=False,
            cbar_kws={"label": "% mujeres", "shrink": 0.5})
ax.set_title("Ratio de feminización migratoria por provincia y año\n"
             "(azul > 50% = más mujeres · rojo < 50% = más hombres)", pad=12)
ax.set_xlabel("Año")
ax.set_ylabel("")
ax.tick_params(axis="y", labelsize=8)

plt.tight_layout()
plt.savefig(FIGS / "04_heatmap_feminizacion_provincia_anio.png")
plt.show()
print("✅ 04_heatmap_feminizacion_provincia_anio.png")

In [ ]:
#============================================================
# Celda 8 — Scatter: feminización vs saldo neto (¿atraen más las provincias feminizadas?)
#============================================================
from scipy import stats

media2 = df_gen.groupby("provincia")[
    ["ratio_feminizacion", "saldo_neto", "cob_100mbps_pct"]
].mean().reset_index()

slope, intercept, r, p, _ = stats.linregress(
    media2["ratio_feminizacion"], media2["saldo_neto"]
)
x_line = np.linspace(media2["ratio_feminizacion"].min(),
                     media2["ratio_feminizacion"].max(), 100)

fig, ax = plt.subplots(figsize=(11, 7))
sc = ax.scatter(media2["ratio_feminizacion"], media2["saldo_neto"],
                c=media2["cob_100mbps_pct"], cmap="YlGn",
                s=80, edgecolors="white", alpha=0.85, zorder=3)
plt.colorbar(sc, ax=ax, label="Cobertura 100Mbps (%)")

ax.plot(x_line, slope * x_line + intercept,
        color=C_NEU, linewidth=2, linestyle="--",
        label=f"Tendencia (r={r:.2f}, p={p:.3f})")

# Anotar destacados
DESTACAR = ["Madrid", "Barcelona", "Málaga", "Soria", "Teruel",
            "Islas Baleares", "Alicante/Alacant", "Guadalajara"]
for _, row in media2.iterrows():
    if row["provincia"] in DESTACAR:
        ax.annotate(row["provincia"],
                    xy=(row["ratio_feminizacion"], row["saldo_neto"]),
                    xytext=(5, 4), textcoords="offset points",
                    fontsize=7.5, color="#2c3e50")

ax.axhline(0, color="black", linewidth=0.7, linestyle=":")
ax.axvline(50, color="black", linewidth=0.7, linestyle=":", alpha=0.5)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_xlabel("Ratio feminización migratoria (%)")
ax.set_ylabel("Saldo neto migratorio medio")
ax.set_title("¿Las provincias con migración más feminizada tienen mayor atracción?\n"
             "(color = cobertura 100Mbps)")
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig(FIGS / "05_scatter_feminizacion_saldo.png")
plt.show()
print("✅ 05_scatter_feminizacion_saldo.png")

In [ ]:
#============================================================
# Celda 9 — Brecha absoluta media por provincia
#============================================================
brecha = (df_gen.groupby("provincia")["brecha_absoluta"]
          .mean().reset_index()
          .sort_values("brecha_absoluta", ascending=True))

colores = [C_M if v >= 0 else C_H for v in brecha["brecha_absoluta"]]

fig, ax = plt.subplots(figsize=(10, 16))
ax.barh(brecha["provincia"], brecha["brecha_absoluta"],
        color=colores, edgecolor="white")
ax.axvline(0, color="black", linewidth=0.9)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
ax.set_title("Brecha absoluta migratoria por provincia\n"
             "(Mujeres − Hombres · media 2013–2021)\n"
             "Rojo = más mujeres · Azul = más hombres",
             fontsize=11)
ax.set_xlabel("Diferencia (personas)")
ax.tick_params(axis="y", labelsize=8)

plt.tight_layout()
plt.savefig(FIGS / "06_brecha_absoluta_provincias.png")
plt.show()
print("✅ 06_brecha_absoluta_provincias.png")

In [ ]:
#============================================================
# Celda 10 — Tabla resumen estadístico
#============================================================
resumen = df_gen.groupby("anyo").agg(
    flujo_hombres=("flujo_hombres", "sum"),
    flujo_mujeres=("flujo_mujeres", "sum"),
    ratio_fem_media=("ratio_feminizacion", "mean"),
    brecha_media=("brecha_absoluta", "mean"),
).reset_index()

resumen["flujo_total"] = resumen["flujo_hombres"] + resumen["flujo_mujeres"]
resumen = resumen.round(2)

print("📊 Resumen nacional flujos por sexo (2013–2021)\n")
print(resumen.to_string(index=False))
resumen.to_csv(TABLES / "resumen_nacional_sexo.csv", index=False)

# Tabla ranking feminización por provincia
ranking_fem = (df_gen.groupby("provincia")
               [["ratio_feminizacion", "brecha_absoluta", "saldo_neto"]]
               .mean().round(2).reset_index()
               .sort_values("ratio_feminizacion", ascending=False))
ranking_fem.to_csv(TABLES / "ranking_feminizacion_provincia.csv", index=False)
print("\n✅ Tablas exportadas")

In [ ]:
#============================================================
# Celda 11 — Cierre
#============================================================
print("=" * 55)
print("🏁 09_analisis_genero completado")
print("=" * 55)
print(f"\n📁 Figuras  → {FIGS}")
print(f"📁 Tablas   → {TABLES}")
print("\n📈 Figuras:")
for f in sorted(FIGS.glob("*.png")):
    print(f"   ✅ {f.name}")
print("\n📋 Tablas:")
for f in sorted(TABLES.glob("*.csv")):
    print(f"   ✅ {f.name}")